# Diamond Price Prediction and Market Segmentation
## Phase 3 - Outlier Handling and Skewness Treatment

This notebook performs:
- focused outlier treatment for `carat`, `price`, `x`, `y`, `z`
- skewness analysis and transformation testing
- before/after comparison of outlier counts and skewness
- saving interim datasets and preprocessing figures

This notebook uses the project utilities already defined in:
- `src.utils.paths`
- `src.utils.config`
- `src.utils.io`
- `src.data.load_data`
- `src.data.clean_data`
- `src.eda.univariate`
- `src.visualization.plot_eda`

## Imports
Load project utilities, EDA helpers, and plotting functions.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy import stats

from src.utils.paths import find_project_root, resolve_project_path
from src.utils.config import load_project_configs
from src.utils.io import ensure_dir, save_csv_file
from src.utils.logger import get_logger

from src.data.clean_data import clean_diamonds_dataset

from src.eda.univariate import (
    compute_iqr_bounds,
    count_iqr_outliers,
    build_outlier_summary,
    clip_outliers_iqr,
    compute_skewness_table,
    find_high_skew_columns,
    evaluate_skew_transforms,
    apply_selected_transformations
)

from src.visualization.plot_eda import plot_outlier_boxplots_before_after, plot_skewness_before_after

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

## Resolve project root and load configurations
Use the same repository-aware path resolution used in earlier phases.

In [2]:
project_root = find_project_root()
configs = load_project_configs(project_root)

main_config = configs["main_config"]
paths_config = configs["paths_config"]
features_config = configs["features_config"]

logger = get_logger("phase03.outlier_skewness")

project_root

WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation')

## Load cleaned input dataset
This phase should start from the cleaned dataset generated in the previous preprocessing phase.
If it does not exist yet, regenerate it using `clean_diamonds_dataset()`.

In [3]:
cleaned_input_path = resolve_project_path(project_root, "data/processed/diamonds_processed.csv")

if not Path(cleaned_input_path).exists():
    previous_phase_results = clean_diamonds_dataset(project_root)
    print("Generated cleaned input from previous phase:", previous_phase_results["processed_output"])

df = pd.read_csv(cleaned_input_path)
print("Input shape:", df.shape)
display(df.head())

Input shape: (53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


## Define focused columns for this phase
Outlier treatment and skewness treatment will be focused on:
- `carat`
- `price`
- `x`
- `y`
- `z`

In [4]:
focus_numeric_cols = ["carat", "price", "x", "y", "z"]
display(df[focus_numeric_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
carat,53940.0,0.797940,0.474011,0.20,0.40,0.70,1.04,5.01
price,53940.0,3932.799722,3989.439738,326.00,950.00,2401.00,5324.25,18823.00
x,53940.0,5.732003,1.119587,3.73,4.71,5.70,6.54,10.74
y,53940.0,5.735267,1.140265,3.68,4.72,5.71,6.54,58.90
z,53940.0,3.540043,0.702400,1.07,2.91,3.53,4.04,31.80


## Build outlier summary before treatment
Use the IQR method because it is robust for heavy-tailed economic and size-related variables.
This notebook will clip outliers instead of removing rows so that the dataset size remains stable.

In [5]:
outlier_summary_before = build_outlier_summary(df = df, columns = focus_numeric_cols)

display(outlier_summary_before)
print("Total focused outliers before treatment:", int(outlier_summary_before["outlier_count"].sum()))

,column,non_null_count,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,carat,53940,0.40,1.04,0.64,-0.560,2.000,1889,3.5020
1,price,53940,950.00,5324.25,4374.25,-5611.375,11885.625,3540,6.5628
2,x,53940,4.71,6.54,1.83,1.965,9.285,24,0.0445
3,y,53940,4.72,6.54,1.82,1.990,9.270,22,0.0408
4,z,53940,2.91,4.04,1.13,1.215,5.735,29,0.0538


Total focused outliers before treatment: 5504


## Visual inspection before outlier treatment
Create boxplots for the focused numeric columns before clipping.

In [6]:
preprocessing_fig_dir = resolve_project_path(project_root, "figures/preprocessing")
ensure_dir(preprocessing_fig_dir)

outlier_before_path = resolve_project_path(project_root, "figures/preprocessing/outlier_boxplot_before.png")

plot_outlier_boxplots_before_after(before_df = df, after_df = None, columns = focus_numeric_cols, before_path = outlier_before_path, after_path = None, mode = "before_only")

print("Saved:", outlier_before_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\preprocessing\outlier_boxplot_before.png


## Apply IQR clipping
Clip values to the lower and upper IQR bounds column-wise.
This keeps all rows while reducing extreme influence on downstream regression and clustering.

In [7]:
df_outliers_treated, outlier_bounds_df = clip_outliers_iqr(df = df, columns = focus_numeric_cols)

display(outlier_bounds_df)
display(df_outliers_treated[focus_numeric_cols].describe().T)

,column,lower_bound,upper_bound
0,carat,-0.560,2.000
1,price,-5611.375,11885.625
2,x,1.965,9.285
3,y,1.990,9.270
4,z,1.215,5.735


,count,mean,std,min,25%,50%,75%,max
carat,53940.0,0.792558,0.457089,0.200,0.40,0.70,1.04,2.000
price,53940.0,3732.145690,3436.769344,326.000,950.00,2401.00,5324.25,11885.625
x,53940.0,5.731839,1.119016,3.730,4.71,5.70,6.54,9.285
y,53940.0,5.733793,1.111132,3.680,4.72,5.71,6.54,9.270
z,53940.0,3.539359,0.691047,1.215,2.91,3.53,4.04,5.735


## Compare outlier counts before and after treatment
Count outliers again after clipping to verify the effect.

In [8]:
outlier_summary_after = build_outlier_summary(
    df=df_outliers_treated,
    columns=focus_numeric_cols,
)

outlier_comparison = outlier_summary_before.merge(
    outlier_summary_after[["column", "outlier_count"]],
    on="column",
    suffixes=("_before", "_after"),
)

outlier_comparison["reduction"] = (
    outlier_comparison["outlier_count_before"] - outlier_comparison["outlier_count_after"]
)

display(outlier_comparison)

,column,non_null_count,q1,q3,iqr,lower_bound,upper_bound,outlier_count_before,outlier_pct,outlier_count_after,reduction
0,carat,53940,0.40,1.04,0.64,-0.560,2.000,1889,3.5020,0,1889
1,price,53940,950.00,5324.25,4374.25,-5611.375,11885.625,3540,6.5628,0,3540
2,x,53940,4.71,6.54,1.83,1.965,9.285,24,0.0445,0,24
3,y,53940,4.72,6.54,1.82,1.990,9.270,22,0.0408,0,22
4,z,53940,2.91,4.04,1.13,1.215,5.735,29,0.0538,0,29


## Visual inspection after outlier treatment
Create boxplots after clipping for comparison.

In [9]:
outlier_after_path = resolve_project_path(
    project_root,
    "figures/preprocessing/outlier_boxplot_after.png"
)

plot_outlier_boxplots_before_after(
    before_df=df,
    after_df=df_outliers_treated,
    columns=focus_numeric_cols,
    before_path=None,
    after_path=outlier_after_path,
    mode="after_only",
)

print("Saved:", outlier_after_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\preprocessing\outlier_boxplot_after.png


## Document final outlier decision
Final decision:
- method used: **IQR**
- action used: **clip to lower/upper bounds**
- reason: preserves row count while controlling extreme values in highly skewed numeric variables

In [10]:
outlier_decision = {
    "method": "IQR",
    "action": "clip",
    "columns": focus_numeric_cols,
    "reason": "Robust to skewed numeric distributions and preserves dataset size for downstream modeling.",
}

outlier_decision

{'method': 'IQR',
 'action': 'clip',
 'columns': ['carat', 'price', 'x', 'y', 'z'],
 'reason': 'Robust to skewed numeric distributions and preserves dataset size for downstream modeling.'}

## Compute skewness before transformation
Calculate skewness on the outlier-treated data so the transformation decision is based on stabilized numeric ranges.

In [11]:
skewness_before = compute_skewness_table(
    df=df_outliers_treated,
    columns=focus_numeric_cols,
)

display(skewness_before)

,column,skewness,abs_skewness
0,price,1.148304,1.148304
1,carat,0.899893,0.899893
2,x,0.394179,0.394179
3,y,0.389861,0.389861
4,z,0.387198,0.387198


## Identify highly skewed variables
Columns with absolute skewness greater than the configured threshold will be tested with multiple transformations.

In [12]:
skew_threshold = 1.0

high_skew_cols = find_high_skew_columns(
    skewness_df=skewness_before,
    threshold=skew_threshold,
)

print("Highly skewed columns:", high_skew_cols)

Highly skewed columns: ['price']


## Test candidate transformations
For each highly skewed variable, compare:
- `log1p`
- `sqrt`
- `boxcox` if all values are strictly positive

The best transformation is the one that produces the lowest absolute skewness.

In [13]:
transform_evaluation = evaluate_skew_transforms(
    df=df_outliers_treated,
    columns=high_skew_cols,
)

display(transform_evaluation)

,column,original_skewness,log1p_abs_skewness,sqrt_abs_skewness,boxcox_abs_skewness,boxcox_lambda,selected_transformation,selected_abs_skewness
0,price,1.148304,0.031771,0.61224,0.008054,-0.019999,boxcox,0.008054


## Apply selected skewness transformations
Apply the best transformation chosen from the evaluation table.

In [14]:
df_skewness_treated, selected_transformations = apply_selected_transformations(
    df=df_outliers_treated,
    transform_evaluation_df=transform_evaluation,
)

display(selected_transformations)
display(df_skewness_treated[focus_numeric_cols].head())

,column,selected_transformation,boxcox_lambda
0,price,boxcox,-0.019999


,carat,price,x,y,z
0,0.23,5.464589,3.95,3.98,2.43
1,0.21,5.464589,3.89,3.84,2.31
2,0.23,5.467317,4.05,4.07,2.31
3,0.29,5.486178,4.20,4.23,2.63
4,0.31,5.488840,4.34,4.35,2.75


## Compare skewness before and after treatment
This confirms whether the selected transforms reduced skewness successfully.

In [15]:
skewness_after = compute_skewness_table(
    df=df_skewness_treated,
    columns=focus_numeric_cols,
)

skewness_comparison = skewness_before.merge(
    skewness_after[["column", "skewness"]],
    on="column",
    suffixes=("_before", "_after"),
)

skewness_comparison["abs_before"] = skewness_comparison["skewness_before"].abs()
skewness_comparison["abs_after"] = skewness_comparison["skewness_after"].abs()
skewness_comparison["improvement"] = (
    skewness_comparison["abs_before"] - skewness_comparison["abs_after"]
)

display(skewness_comparison.sort_values("abs_before", ascending=False))

,column,skewness_before,abs_skewness,skewness_after,abs_before,abs_after,improvement
0,price,1.148304,1.148304,0.008054,1.148304,0.008054,1.14025
1,carat,0.899893,0.899893,0.899893,0.899893,0.899893,0.00000
2,x,0.394179,0.394179,0.394179,0.394179,0.394179,0.00000
3,y,0.389861,0.389861,0.389861,0.389861,0.389861,0.00000
4,z,0.387198,0.387198,0.387198,0.387198,0.387198,0.00000


## Plot skewness before and after treatment
Save comparison barplots for quick documentation.

In [16]:
skew_before_path = resolve_project_path(
    project_root,
    "figures/preprocessing/skewness_before.png"
)
skew_after_path = resolve_project_path(
    project_root,
    "figures/preprocessing/skewness_after.png"
)

plot_skewness_before_after(
    skew_before_df=skewness_before,
    skew_after_df=skewness_after,
    before_path=skew_before_path,
    after_path=skew_after_path,
)

print("Saved:", skew_before_path)
print("Saved:", skew_after_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\preprocessing\skewness_before.png
Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\preprocessing\skewness_after.png


## Save interim outputs
Persist the outlier-treated and skewness-treated datasets for the next notebook.

In [17]:
outliers_treated_path = resolve_project_path(
    project_root,
    "data/interim/diamonds_outliers_treated.csv"
)
skewness_treated_path = resolve_project_path(
    project_root,
    "data/interim/diamonds_skewness_treated.csv"
)

save_csv_file(df_outliers_treated, outliers_treated_path, index=False)
save_csv_file(df_skewness_treated, skewness_treated_path, index=False)

print("Saved:", outliers_treated_path)
print("Saved:", skewness_treated_path)

Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\interim\diamonds_outliers_treated.csv
Saved: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\interim\diamonds_skewness_treated.csv


## Phase summary
- Outlier handling completed on `carat`, `price`, `x`, `y`, `z`
- Outlier counts compared before and after clipping
- Skewness computed and reduced using tested transformations
- Interim datasets and preprocessing figures saved successfully

In [18]:
summary_payload = {
    "input_shape": df.shape,
    "outliers_treated_shape": df_outliers_treated.shape,
    "skewness_treated_shape": df_skewness_treated.shape,
    "selected_transformations": selected_transformations.to_dict(orient="records"),
}

summary_payload

{'input_shape': (53940, 10),
 'outliers_treated_shape': (53940, 10),
 'skewness_treated_shape': (53940, 10),
 'selected_transformations': [{'column': 'price',
   'selected_transformation': 'boxcox',
   'boxcox_lambda': -0.01999873349402444}]}